# Module 2 — Scaling, Leakage & the Confusion Matrix
**Lectures:** Week 4 (scaling: StandardScaler/MinMax/Robust, fit on train only, confusion matrix)

Three silent bugs that make real ML projects fail in production. You'll cause each one, then fix it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ieso_lab import load_dataset
from ieso_lab.features import add_peak_label
from ieso_lab.tracking import log_run, show_runs

plt.rcParams["figure.figsize"] = (9, 4)
df = load_dataset().dropna().reset_index(drop=True)
print(df.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
FEATURES = ["hour", "temperature_2m", "relative_humidity_2m", "is_weekend"]
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
thr = train_df["demand_mw"].quantile(0.90)
train_df, test_df = add_peak_label(train_df.copy(), thr), add_peak_label(test_df.copy(), thr)
X_train, y_train = train_df[FEATURES], train_df["is_peak"]
X_test, y_test = test_df[FEATURES], test_df["is_peak"]
X_train.describe().loc[["min", "max", "std"]]

## 🔨 Bug 1: unscaled distances
Humidity spans 0–100, `is_weekend` spans 0–1. **Predict:** in a Euclidean distance, how much does weekend vs weekday matter?

In [ ]:
a, b = X_train.iloc[0].values, X_train.iloc[1].values
contrib = (a - b) ** 2
print(pd.Series(contrib / contrib.sum(), index=FEATURES).round(3), "← share of squared distance")

In [ ]:
# TODO 6: fit a StandardScaler on X_train ONLY, transform both sets
scaler = None
X_train_s = None
X_test_s = None

raw = KNeighborsClassifier(n_neighbors=15).fit(X_train, y_train).score(X_test, y_test)
scaled = KNeighborsClassifier(n_neighbors=15).fit(X_train_s, y_train).score(X_test_s, y_test)
print(f"unscaled: {raw:.4f} | scaled: {scaled:.4f}")

The math: $z = \frac{x-\mu}{\sigma}$ puts every feature in "standard deviations from its mean", so each one gets a comparable vote in the distance.

## 🔨 Bug 2: leakage from a random split on time series
Hour 14:00 and 15:00 on the same day are almost identical. A random split puts one in train and one in test, so the model "sees the answer". Real forecasting = train on the **past**, test on the **future**.

In [ ]:
df_sorted = df.sort_values("timestamp_utc").reset_index(drop=True)
# TODO 7: time-based split: first 80% of rows = train, last 20% = test (no shuffling)
cut = None
tr_t, te_t = None, None

In [ ]:
thr_t = tr_t["demand_mw"].quantile(0.90)
tr_t, te_t = add_peak_label(tr_t.copy(), thr_t), add_peak_label(te_t.copy(), thr_t)
LAGF = FEATURES + ["demand_lag_24h"]
def knn_score(tr, te, feats):
    sc = StandardScaler().fit(tr[feats])
    m = KNeighborsClassifier(n_neighbors=15).fit(sc.transform(tr[feats]), tr["is_peak"])
    return m.score(sc.transform(te[feats]), te["is_peak"])
print("random split:", round(knn_score(train_df, test_df, LAGF), 4))
print("time split:  ", round(knn_score(tr_t, te_t, LAGF), 4))
print("test period:", te_t.timestamp_utc.min().date(), "→", te_t.timestamp_utc.max().date())

**Bug 3 (think, don't code):** if you call `scaler.fit(X)` on *all* data before splitting, the test set's mean/std leak into training. Same for picking the peak threshold from all data. Rule: **anything learned from data is learned from train only.**

## Confusion matrix (Week 4)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
sc = StandardScaler().fit(tr_t[LAGF])
model = KNeighborsClassifier(n_neighbors=15).fit(sc.transform(tr_t[LAGF]), tr_t["is_peak"])
y_pred = model.predict(sc.transform(te_t[LAGF]))
# TODO 8: build the confusion matrix (labels [0, 1]) and display it
cm = None

In [ ]:
tn, fp, fn, tp = cm.ravel()
print(f"Missed peaks (FN): {fn} | False alarms (FP): {fp}")
log_run("02", "knn_scaled_timesplit", {"k": 15, "features": LAGF}, {"accuracy": (tn + tp) / cm.sum()})

## Why questions
1. Why did scaling change (or not change) KNN accuracy here? Use the distance-share table.
2. Why is the time-split score more honest? Which one would you report to a manager?
3. For a factory avoiding peak charges, which is worse: FN or FP? Why?

---
## 🔧 Git task (do this before you start, and finish after)
```bash
git switch main && git pull
git switch -c module-02-scaling          # start of module
# ... work ...
git add notebooks/ experiments/runs.csv
git commit -m "Module 2: scaling, time split, confusion matrix"
git push -u origin module-02-scaling
```
Then open a **Pull Request** on GitHub → read your own diff → merge → close the module's Issue.

**Git skill this module:** before merging, run `git log --oneline` and `git diff main` — read what changed.